# Customer Churn Forecasting - Model Comparison
## Module 7: Side-by-Side Model Benchmarking

Comprehensive comparison of all trained models (Logistic Regression, Decision Tree, KNN, Random Forest + SMOTE) across multiple evaluation metrics.

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, roc_curve, classification_report)
try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'imbalanced-learn', '-q'])
    from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('All libraries loaded successfully.')

## 7.1 Prepare Data (Unified Pipeline)

In [ ]:
# Load and prepare the dataset
if os.path.exists('WA_Fn-UseC_-Telco-Customer-Churn.csv'):
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
elif os.path.exists('telco_customer_churn.csv'):
    df = pd.read_csv('telco_customer_churn.csv')
else:
    raise FileNotFoundError('Dataset not found.')

# Full preprocessing pipeline
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['tenure'] * df['MonthlyCharges'], inplace=True)
df.drop(columns=['customerID'], inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Feature engineering
df['AvgMonthlySpend'] = np.where(df['tenure'] > 0, df['TotalCharges'] / df['tenure'], df['MonthlyCharges'])
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['TotalServices'] = sum((df[col].isin(['Yes', 'Fiber optic', 'DSL'])).astype(int) for col in service_cols if col in df.columns)
df['ChargesPerService'] = np.where(df['TotalServices'] > 0, df['MonthlyCharges'] / df['TotalServices'], df['MonthlyCharges'])
contract_risk = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
df['ContractRisk'] = df['Contract'].map(contract_risk)

cat_cols = df.select_dtypes(include=['object']).columns.tolist()
binary_cols = [col for col in cat_cols if df[col].nunique() == 2]
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])
multi_cat_cols = [col for col in cat_cols if df[col].nunique() > 2]
df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True, dtype=int)

X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SMOTE for Random Forest
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f'Data prepared. Train: {len(X_train):,} | Test: {len(X_test):,}')

## 7.2 Train All Models

In [ ]:
# Train all models
models = {}

# 1. Logistic Regression (scaled data, balanced weights)
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_scaled, y_train)
models['Logistic Regression'] = {
    'model': lr,
    'y_pred': lr.predict(X_test_scaled),
    'y_prob': lr.predict_proba(X_test_scaled)[:, 1]
}

# 2. Decision Tree (balanced weights)
dt = DecisionTreeClassifier(max_depth=6, min_samples_split=20, min_samples_leaf=10,
                            class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)
models['Decision Tree'] = {
    'model': dt,
    'y_pred': dt.predict(X_test),
    'y_prob': dt.predict_proba(X_test)[:, 1]
}

# 3. K-Nearest Neighbors (scaled data)
knn = KNeighborsClassifier(n_neighbors=7, weights='distance', n_jobs=-1)
knn.fit(X_train_scaled, y_train)
models['KNN (K=7)'] = {
    'model': knn,
    'y_pred': knn.predict(X_test_scaled),
    'y_prob': knn.predict_proba(X_test_scaled)[:, 1]
}

# 4. Random Forest + SMOTE
rf = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_split=10,
                            min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train_smote, y_train_smote)
models['Random Forest + SMOTE'] = {
    'model': rf,
    'y_pred': rf.predict(X_test),
    'y_prob': rf.predict_proba(X_test)[:, 1]
}

print('All models trained successfully.')

## 7.3 Model Performance Comparison Table

In [ ]:
# Create comparison DataFrame
results = []
for name, m in models.items():
    results.append({
        'Model': name,
        'Accuracy (%)': accuracy_score(y_test, m['y_pred']) * 100,
        'Precision (%)': precision_score(y_test, m['y_pred']) * 100,
        'Recall (%)': recall_score(y_test, m['y_pred']) * 100,
        'F1-Score': f1_score(y_test, m['y_pred']),
        'ROC-AUC': roc_auc_score(y_test, m['y_prob'])
    })

comparison_df = pd.DataFrame(results).set_index('Model')
comparison_df = comparison_df.round(4)

print('=' * 80)
print('MODEL COMPARISON - PERFORMANCE METRICS')
print('=' * 80)
display(comparison_df)

# Highlight best per metric
print('\nBest Model per Metric:')
for col in comparison_df.columns:
    best = comparison_df[col].idxmax()
    print(f'  {col}: {best} ({comparison_df.loc[best, col]:.4f})')

## 7.4 Visual Comparison

In [ ]:
# Bar chart comparison
metrics_to_plot = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score', 'ROC-AUC']
fig, axes = plt.subplots(1, 5, figsize=(25, 6))

colors_bar = ['#3498db', '#e67e22', '#27ae60', '#8e44ad']

for i, metric in enumerate(metrics_to_plot):
    values = comparison_df[metric].values
    bars = axes[i].bar(range(len(values)), values, color=colors_bar, edgecolor='black', alpha=0.85)
    axes[i].set_title(metric, fontsize=12, fontweight='bold')
    axes[i].set_xticks(range(len(values)))
    axes[i].set_xticklabels(['LR', 'DT', 'KNN', 'RF+SMOTE'], rotation=45, fontsize=9)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                     f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()

## 7.5 ROC Curve Comparison

In [ ]:
# Combined ROC Curves
plt.figure(figsize=(10, 7))

colors_roc = {'Logistic Regression': '#3498db', 'Decision Tree': '#e67e22', 
              'KNN (K=7)': '#27ae60', 'Random Forest + SMOTE': '#8e44ad'}

for name, m in models.items():
    fpr, tpr, _ = roc_curve(y_test, m['y_prob'])
    auc = roc_auc_score(y_test, m['y_prob'])
    plt.plot(fpr, tpr, color=colors_roc[name], lw=2, label=f'{name} (AUC = {auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier (AUC = 0.5000)')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC Curve Comparison - All Models', fontsize=15, fontweight='bold')
plt.legend(fontsize=11, loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7.6 Radar Chart - Multi-Metric Comparison

In [ ]:
# Radar chart for multi-metric comparison
from matplotlib.patches import FancyBboxPatch

categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
model_names = list(models.keys())

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(polar=True))

# Normalize all metrics to 0-1 scale
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

for idx, name in enumerate(model_names):
    m = models[name]
    values = [
        accuracy_score(y_test, m['y_pred']),
        precision_score(y_test, m['y_pred']),
        recall_score(y_test, m['y_pred']),
        f1_score(y_test, m['y_pred']),
        roc_auc_score(y_test, m['y_prob'])
    ]
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, label=name, color=list(colors_roc.values())[idx])
    ax.fill(angles, values, alpha=0.1, color=list(colors_roc.values())[idx])

ax.set_yticklabels([])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_title('Multi-Metric Model Comparison (Radar)', fontsize=14, fontweight='bold', pad=20)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
plt.tight_layout()
plt.savefig('model_radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7.7 Key Comparison Insights

### Model Selection Recommendations:
1. **Best Overall Balance:** Random Forest + SMOTE delivers the best F1-Score and balanced precision-recall
2. **Highest Churn Capture (Recall):** Logistic Regression with balanced weights catches the most churners
3. **Most Interpretable:** Decision Tree provides clear, explainable decision rules
4. **Instance-Based Baseline:** KNN provides competitive performance with minimal assumptions

### Business Recommendation:
- **For maximum churn prevention:** Use Logistic Regression (high recall minimizes missed churners)
- **For balanced operations:** Use Random Forest + SMOTE (best F1 and ROC-AUC)
- **For explainability to stakeholders:** Use Decision Tree (clear if-then rules)